# 08b — EDA phần B: KHI NÀO (người phụ trách: Bengaluru/Amritsar)

**Câu hỏi nghiên cứu chung**

> **Ô nhiễm không khí nặng (AQI từ 201 trở lên) tập trung ở thành phố nào và vào mùa nào trong 8 thành phố Ấn Độ giai đoạn 2015–2020, để cơ quan môi trường và người dân biết nên ưu tiên cảnh báo sức khỏe và giảm phát thải ở đâu, lúc nào?**


**Quy ước chung** (chi tiết ở `08_eda.ipynb`): so sánh giữa thành phố chỉ dùng `win` (khoảng chung); "nặng" = AQI >= 201 trên số ngày **có AQI**;
mùa: Đông (12-2), Hè (3-5), Mưa (6-9), Sau mưa (10-11); chỉ viết "có liên quan", **không** viết "gây ra".
Hàm dùng chung nằm ở `src/eda_utils.py` (không sửa nếu chưa báo nhóm).

**Làm việc nhóm:** chỉ sửa notebook **này** và chỉ lưu hình có tên của phần mình (`eda_<số>_...png`). `git pull` trước khi push.

## Việc cần làm
1. **Biểu đồ 2**: heatmap thành phố x tháng (AQI trung vị, thang màu tuần tự, ghi số trên ô) để thấy mùa cao điểm.
2. **Biểu đồ 3**: xu hướng theo năm **trong từng thành phố**, chỉ tháng 1-6 (cùng kỳ) để năm 2020 so được với các năm trước.
3. Chẩn đoán: liệt kê confounder cho mỗi phát hiện (mùa, xu hướng dài hạn, lockdown 2020, thay đổi trạm đo và độ phủ dữ liệu).

## Cẩn thận (đã phát hiện khi kiểm tra)
- Chuỗi các thành phố dài ngắn khác nhau và có vài ô **thưa dữ liệu**: ví dụ Jaipur tháng 1-6/2017 chỉ có 11 ngày có AQI, Gurugram 2016 chỉ 95 ngày. Hãy **luôn xem bảng số ngày `n_tbl`** cạnh bảng trung bình trước khi kết luận về xu hướng, và bỏ hoặc ghi chú các ô có ít ngày.
- Năm 2020 chỉ đến 01/07 và có lockdown: chỉ nêu "có liên quan", không kết luận nguyên nhân. Mùa, xu hướng giảm dần qua các năm và lockdown đều là confounder của việc "2020 thấp hơn 2019".
- Heatmap dùng `win` (khoảng chung) vì cần so các thành phố; biểu đồ xu hướng dùng `eda` nhưng **trong từng thành phố riêng**.

## Chuẩn biểu đồ (W7), tick đủ trước khi push

- [ ] Tiêu đề là **kết luận có con số** (không phải "AQI theo thành phố")
- [ ] Trục có **nhãn và đơn vị**
- [ ] Biểu đồ cột **bắt đầu từ 0**, sắp theo giá trị (không theo bảng chữ cái)
- [ ] **Một** màu nhấn cho điều muốn người đọc thấy, còn lại màu trung tính
- [ ] Ghi nhãn trực tiếp, bỏ legend nếu chỉ có một nhóm
- [ ] Không dùng biểu đồ tròn, 3D, trục kép; không dùng chỉ đỏ-xanh lá để phân biệt
- [ ] Thử chuyển sang thang xám vẫn đọc được
- [ ] Chú thích có cỡ mẫu và ngày dữ liệu (hàm `finish_chart` đã làm sẵn)

## 0. Setup

In [1]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from eda_utils import (CITIES, USABLE, SEASON_ORDER, ACCENT, NEUTRAL,
                       vn, load, poor_share, four_families, finish_chart, save_fig, set_style)

set_style()
eda, win, START, END = load()      # eda: toàn bộ 2015-2020 | win: khoảng chung của 8 thành phố (dùng khi so sánh giữa thành phố)

Toàn bộ: 14.056 dòng, 01/01/2015 - 01/07/2020
Khoảng chung (biến win): 14/06/2017 - 01/07/2020, 8.912 dòng; 8.734 ngày có AQI


## B0. Bảng xu hướng tháng 1-6, kèm số ngày để biết ô nào thưa

In [2]:
h1 = eda[eda.month <= 6].dropna(subset=["AQI"])
trend_tbl = h1.pivot_table(index="City", columns="year", values="AQI", aggfunc="mean", observed=True).round(0)
n_tbl     = h1.pivot_table(index="City", columns="year", values="AQI", aggfunc="size", observed=True)
print("AQI trung bình, tháng 1-6:");  display(trend_tbl)
print("Số ngày có AQI trong ô đó (ô nhỏ = kết luận yếu):"); display(n_tbl)

AQI trung bình, tháng 1-6:


year,2015,2016,2017,2018,2019,2020
City,,,,,,
Delhi,306.0,298.0,294.0,263.0,246.0,182.0
Gurugram,NaN,239.0,319.0,250.0,211.0,159.0
Lucknow,133.0,257.0,260.0,257.0,224.0,158.0
Jaipur,NaN,NaN,83.0,165.0,132.0,105.0
Bengaluru,111.0,114.0,89.0,86.0,108.0,80.0
Amritsar,NaN,NaN,92.0,148.0,104.0,93.0
Chennai,178.0,138.0,109.0,105.0,109.0,80.0
Hyderabad,135.0,127.0,128.0,105.0,104.0,78.0


Số ngày có AQI trong ô đó (ô nhỏ = kết luận yếu):


year,2015,2016,2017,2018,2019,2020
City,,,,,,
Delhi,181.0,182.0,180.0,181.0,181.0,182.0
Gurugram,NaN,95.0,161.0,163.0,180.0,182.0
Lucknow,98.0,160.0,178.0,181.0,181.0,182.0
Jaipur,NaN,NaN,11.0,180.0,181.0,182.0
Bengaluru,102.0,180.0,181.0,177.0,181.0,182.0
Amritsar,NaN,NaN,111.0,156.0,179.0,173.0
Chennai,95.0,169.0,178.0,181.0,181.0,182.0
Hyderabad,86.0,179.0,180.0,181.0,181.0,182.0


## B1. Biểu đồ 2: heatmap thành phố x tháng

In [3]:
# TODO (người B): pivot AQI trung vị theo (City, month) trên `win`, vẽ bằng ax.imshow (thang màu tuần tự),
# ghi số trên ô, lưu save_fig(fig, "02_heatmap_city_month")

## B2. Biểu đồ 3: xu hướng theo năm trong từng thành phố (tháng 1-6)

In [4]:
# TODO (người B): đường theo năm cho từng thành phố (đường là phù hợp vì thể hiện thay đổi theo thời gian),
# nhấn 1-2 thành phố, ghi chú n ngày, lưu save_fig(fig, "03_trend_city_year")

## Phát hiện của phần B (người ghép báo cáo sẽ gom bảng này vào mục 5)

| # | Phát hiện (có con số) | Biểu đồ | Confounder có thể | Hạn chế |
|---|---|---|---|---|
| B1 | | 2 | | |
| B2 | | 3 | | |

**Trước khi push:** Restart & Run All không lỗi; đã tick checklist biểu đồ; đã `git pull`.